# **Segmented Modeling Analysis**
**Hypothesis**: `StoreType` 'b' and `Assortment` 'b' are massive outliers in volume. By training completely distinct models for each Store Type, Assortment, or combination thereof, the trees won't have to waste splits isolating these macro-differences.

**Goal**: Compare Global Random Forest against 3 segmentation strategies.

**Methodology**:
1. Global (1 Model)
2. By `StoreType` (4 Models)
3. By `Assortment` (3 Models)
4. By `StoreType` + `Assortment` (9 Models)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (10, 6)})


# RMSPE Metric
def rmspe(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    y_pred = np.maximum(y_pred, 1.0)
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))


# Load config
CONFIG_PATH = Path("../configs/params.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

## **1. Data Split & Pre-calculated Features**
Logic consistent with notebook 02 for fair comparison and correct methodology.

In [ ]:
# 1. Load Data
train_df = pd.read_csv(f"../{config['data']['raw_train']}", low_memory=False)
store_df = pd.read_csv(f"../{config['data']['raw_store']}")
df = pd.merge(train_df, store_df, on="Store", how="left")
df = df[(df["Open"] == 1) & (df["Sales"] > 0)].copy()

df["Date"] = pd.to_datetime(df["Date"])
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
df.sort_values("Date", inplace=True, ignore_index=True)

max_date = df["Date"].max()
sim_start = max_date - pd.Timedelta(days=config["data_split"]["simulation_days"])
holdout_start = sim_start - pd.Timedelta(days=config["data_split"]["holdout_days"])
df_cv = df[df["Date"] <= holdout_start].copy()


# 2. Build Features Function
def build_features(train_df, test_df):
    df_full = pd.concat([train_df, test_df], axis=0)
    df_full["Year"] = df_full["Date"].dt.year
    df_full["Month"] = df_full["Date"].dt.month
    df_full["WeekOfYear"] = df_full["Date"].dt.isocalendar().week.astype(int)

    # Store raw for segmentation
    df_full["RawStoreType"] = df_full["StoreType"].astype(str)
    df_full["RawAssortment"] = df_full["Assortment"].astype(str)

    df_full["StateHoliday"] = df_full["StateHoliday"].astype(str)
    df_full = pd.get_dummies(
        df_full, columns=["StoreType", "Assortment", "StateHoliday"]
    )

    train_comp_median = train_df["CompetitionDistance"].median()
    df_full["LogCompDist"] = np.log1p(
        df_full["CompetitionDistance"].fillna(train_comp_median)
    )

    train_out = df_full.iloc[: len(train_df)].copy()
    test_out = df_full.iloc[len(train_df) :].copy()

    # Target Encoding
    train_out["Store_TargetMean"] = train_out.groupby("Store")["Sales"].transform(
        lambda x: x.shift().expanding().mean()
    )
    global_mean = train_out["Sales"].mean()
    train_out["Store_TargetMean"].fillna(global_mean, inplace=True)
    test_out["Store_TargetMean"] = (
        test_out["Store"]
        .map(train_out.groupby("Store")["Sales"].mean())
        .fillna(global_mean)
    )

    base_cols = [
        "DayOfWeek",
        "Promo",
        "Year",
        "Month",
        "WeekOfYear",
        "LogCompDist",
        "Store_TargetMean",
    ]
    ohe_cols = [
        c
        for c in train_out.columns
        if any(x in c for x in ["StoreType_", "Assortment_", "StateHoliday_"])
    ]
    meta_cols = ["RawStoreType", "RawAssortment", "Sales"]

    return train_out[base_cols + ohe_cols + meta_cols], test_out[
        base_cols + ohe_cols + meta_cols
    ]


# 3. Pre-calculate Folds
tscv = TimeSeriesSplit(n_splits=config["train"]["n_splits"])
folds_data = []

for tr_idx, te_idx in tscv.split(df_cv):
    X_tr, X_te = build_features(df_cv.iloc[tr_idx], df_cv.iloc[te_idx])
    folds_data.append((X_tr, X_te))

print(f"Pre-calculated {len(folds_data)} temporal folds successfully.")

## **2. Risk Analysis: Data Starvation**
Before modeling, we check the density of these segments. Random Forests are data-hungry. If a segment has too few stores, it won't generalize to seasonal splits well.

In [ ]:
s_counts = (
    df_cv.groupby(["StoreType", "Assortment"])["Store"]
    .nunique()
    .rename("Unique_Stores")
)
r_counts = df_cv.groupby(["StoreType", "Assortment"]).size().rename("Total_Rows")
stats = (
    pd.concat([s_counts, r_counts], axis=1)
    .reset_index()
    .sort_values("Total_Rows", ascending=False)
)

stats["Risk"] = np.where(stats["Unique_Stores"] < 25, "HIGH", "LOW")
display(
    stats.style.apply(
        lambda x: ["background: lightcoral" if v == "HIGH" else "" for v in x],
        subset=["Risk"],
    )
)

<style type="text/css">
#T_32d17_row6_col4, #T_32d17_row7_col4, #T_32d17_row8_col4 {
  background: lightcoral;
}
</style>
<table id="T_32d17">
  <thead>
    <tr>
      <th class="blank level0" >&nbsp;</th>
      <th id="T_32d17_level0_col0" class="col_heading level0 col0" >StoreType</th>
      <th id="T_32d17_level0_col1" class="col_heading level0 col1" >Assortment</th>
      <th id="T_32d17_level0_col2" class="col_heading level0 col2" >Unique_Stores</th>
      <th id="T_32d17_level0_col3" class="col_heading level0 col3" >Total_Rows</th>
      <th id="T_32d17_level0_col4" class="col_heading level0 col4" >Risk</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th id="T_32d17_level0_row0" class="row_heading level0 row0" >0</th>
      <td id="T_32d17_row0_col0" class="data row0 col0" >a</td>
      <td id="T_32d17_row0_col1" class="data row0 col1" >a</td>
      <td id="T_32d17_row0_col2" class="data row0 col2" >381</td>
      <td id="T_32d17_row0_col3" class="data row0 col3" >263833</td>
      <td id="T_32d17_row0_col4" class="data row0 col4" >LOW</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row1" class="row_heading level0 row1" >1</th>
      <td id="T_32d17_row1_col0" class="data row1 col0" >a</td>
      <td id="T_32d17_row1_col1" class="data row1 col1" >c</td>
      <td id="T_32d17_row1_col2" class="data row1 col2" >221</td>
      <td id="T_32d17_row1_col3" class="data row1 col3" >157973</td>
      <td id="T_32d17_row1_col4" class="data row1 col4" >LOW</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row2" class="row_heading level0 row2" >8</th>
      <td id="T_32d17_row2_col0" class="data row2 col0" >d</td>
      <td id="T_32d17_row2_col1" class="data row2 col1" >c</td>
      <td id="T_32d17_row2_col2" class="data row2 col2" >220</td>
      <td id="T_32d17_row2_col3" class="data row2 col3" >152035</td>
      <td id="T_32d17_row2_col4" class="data row2 col4" >LOW</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row3" class="row_heading level0 row3" >7</th>
      <td id="T_32d17_row3_col0" class="data row3 col0" >d</td>
      <td id="T_32d17_row3_col1" class="data row3 col1" >a</td>
      <td id="T_32d17_row3_col2" class="data row3 col2" >128</td>
      <td id="T_32d17_row3_col3" class="data row3 col3" >86427</td>
      <td id="T_32d17_row3_col4" class="data row3 col4" >LOW</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row4" class="row_heading level0 row4" >5</th>
      <td id="T_32d17_row4_col0" class="data row4 col0" >c</td>
      <td id="T_32d17_row4_col1" class="data row4 col1" >a</td>
      <td id="T_32d17_row4_col2" class="data row4 col2" >77</td>
      <td id="T_32d17_row4_col3" class="data row4 col3" >54048</td>
      <td id="T_32d17_row4_col4" class="data row4 col4" >LOW</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row5" class="row_heading level0 row5" >6</th>
      <td id="T_32d17_row5_col0" class="data row5 col0" >c</td>
      <td id="T_32d17_row5_col1" class="data row5 col1" >c</td>
      <td id="T_32d17_row5_col2" class="data row5 col2" >71</td>
      <td id="T_32d17_row5_col3" class="data row5 col3" >50254</td>
      <td id="T_32d17_row5_col4" class="data row5 col4" >LOW</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row6" class="row_heading level0 row6" >3</th>
      <td id="T_32d17_row6_col0" class="data row6 col0" >b</td>
      <td id="T_32d17_row6_col1" class="data row6 col1" >b</td>
      <td id="T_32d17_row6_col2" class="data row6 col2" >9</td>
      <td id="T_32d17_row6_col3" class="data row6 col3" >7579</td>
      <td id="T_32d17_row6_col4" class="data row6 col4" >HIGH</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row7" class="row_heading level0 row7" >2</th>
      <td id="T_32d17_row7_col0" class="data row7 col0" >b</td>
      <td id="T_32d17_row7_col1" class="data row7 col1" >a</td>
      <td id="T_32d17_row7_col2" class="data row7 col2" >7</td>
      <td id="T_32d17_row7_col3" class="data row7 col3" >5919</td>
      <td id="T_32d17_row7_col4" class="data row7 col4" >HIGH</td>
    </tr>
    <tr>
      <th id="T_32d17_level0_row8" class="row_heading level0 row8" >4</th>
      <td id="T_32d17_row8_col0" class="data row8 col0" >b</td>
      <td id="T_32d17_row8_col1" class="data row8 col1" >c</td>
      <td id="T_32d17_row8_col2" class="data row8 col2" >1</td>
      <td id="T_32d17_row8_col3" class="data row8 col3" >872</td>
      <td id="T_32d17_row8_col4" class="data row8 col4" >HIGH</td>
    </tr>
  </tbody>
</table>

*Observation*: `StoreType b` is exceptionally rare. A model trained only on StoreType 'b' with Assortment 'c' would only have 1 single store to learn from. This is extreme starvation risk.

## **3. Segmented Training Engine**
Define a reusable engine to filter data, train models per segment, and recombine predictions into a unified array for scoring.

In [ ]:
rf_params = {
    "n_estimators": 50,
    "max_depth": 10,
    "min_samples_split": 6,
    "random_state": config["train"]["random_state"],
    "n_jobs": -1,
}


def train_segmented(segment_logic, ohe_keep_keys):
    """
    segment_logic: "Global", "StoreType", "Assortment", or "Combo"
    """
    fold_scores = []

    for X_tr, X_te in folds_data:
        # Determine valid features based on segment (drop constants)
        base_cols = [
            "DayOfWeek",
            "Promo",
            "Year",
            "Month",
            "WeekOfYear",
            "LogCompDist",
            "Store_TargetMean",
        ]
        ohe_cols = [c for c in X_tr.columns if any(x in c for x in ohe_keep_keys)]
        features = base_cols + ohe_cols

        unified_trues = []
        unified_preds = []

        # Determine Segments
        if segment_logic == "Global":
            X_tr["Seg"] = "All"
            X_te["Seg"] = "All"
        elif segment_logic == "StoreType":
            X_tr["Seg"] = X_tr["RawStoreType"]
            X_te["Seg"] = X_te["RawStoreType"]
        elif segment_logic == "Assortment":
            X_tr["Seg"] = X_tr["RawAssortment"]
            X_te["Seg"] = X_te["RawAssortment"]
        elif segment_logic == "Combo":
            X_tr["Seg"] = X_tr["RawStoreType"] + "_" + X_tr["RawAssortment"]
            X_te["Seg"] = X_te["RawStoreType"] + "_" + X_te["RawAssortment"]

        for seg in X_tr["Seg"].unique():
            tr_m = X_tr["Seg"] == seg
            te_m = X_te["Seg"] == seg
            if te_m.sum() == 0:
                continue

            # Starvation Fallback
            if tr_m.sum() < 50:
                unified_preds.extend(np.full(te_m.sum(), X_tr[tr_m]["Sales"].mean()))
            else:
                m = RandomForestRegressor(**rf_params)
                m.fit(X_tr[tr_m][features], np.log1p(X_tr[tr_m]["Sales"]))
                unified_preds.extend(np.expm1(m.predict(X_te[te_m][features])))

            unified_trues.extend(X_te[te_m]["Sales"])

        fold_scores.append(rmspe(unified_trues, unified_preds))

    return np.mean(fold_scores)

## **4. Execute Benchmark**

In [ ]:
results = []

# 1. Global
results.append(
    (
        "Global (1 Model)",
        train_segmented("Global", ["StoreType_", "Assortment_", "StateHoliday_"]),
    )
)

# 2. StoreType
results.append(
    (
        "By StoreType (4 Models)",
        train_segmented("StoreType", ["Assortment_", "StateHoliday_"]),
    )
)

# 3. Assortment
results.append(
    (
        "By Assortment (3 Models)",
        train_segmented("Assortment", ["StoreType_", "StateHoliday_"]),
    )
)

# 4. Combo
results.append(("By Combo (9 Models)", train_segmented("Combo", ["StateHoliday_"])))

In [ ]:
# Plot
summary = pd.DataFrame(results, columns=["Strategy", "RMSPE"]).sort_values("RMSPE")

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=summary, x="RMSPE", y="Strategy", palette="magma")
plt.title("Segmented Modeling Leaderboard (Unified RMSPE)")
for p in ax.patches:
    ax.annotate(
        f"{p.get_width() * 100:.2f}%",
        (p.get_width(), p.get_y() + p.get_height() / 2.0),
        ha="left",
        va="center",
        xytext=(5, 0),
        textcoords="offset points",
    )
plt.xlim(0, max(summary["RMSPE"]) * 1.1)
plt.savefig(
    Path("figures") / "07_segmentation_results.png", dpi=300, bbox_inches="tight"
)
plt.show()

## **5. Conclusion: Simplicity Wins**
Our hypothesis proved **false**. Segmenting the models resulted in *worse* unified performance across the board. 

| Strategy | Mean CV RMSPE |
|---|---|
| Global (1 Model) | **22.57%** |
| Segmented: Combo | **23.21%** |
| Segmented: Assortment | **23.75%** |
| Segmented: StoreType | **23.77%** |

#### **Why did this happen?**
1. **Data Starvation**: By slicing by combinations, we created segments with only 1-7 stores. TimeSeries data requires volume to accurately model seasonality rules (like back-to-school or Christmas). The small segments drastically overfit.
2. **Implicit Capacity**: A singular Random Forest/XGBoost is already exceptionally good at segmenting. Providing global `Store_TargetMean` combined with the `StoreType_x` One-Hot Encoded flags allows the global model to natively separate 'b' stores mechanically in its branches, while sharing macro-seasonal rules.

#### **Final Pipeline Decision**
- We reject Model Segregation. 
- We reject Hyperparameter Optuna Tuning (~0.5% gain wasn't worth the compute).
- **The chosen Production Stack**: 1 Global Baseline Model (Random Forest or XGBoost), equipped with Expanding Out-Of-Fold Mean Target Encoding, operating in Log-Target space.